## NB10 — Validação com CEIS e Análise Qualitativa das Propostas de Alto Risco

**Entradas:**
- `data/output/ranking_parcial.csv` — ranking de risco gerado pelo NB09 (75.471 propostas)
- `data/raw/dados_CEIS.csv` — Cadastro de Empresas Inidôneas e Suspensas (CGU, Set/2026)

**Objetivo:**
- Cruzar o ranking de risco com o CEIS para obter um ground truth parcial

---
### 0. Imports e Configuração

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Setup OK")

Setup OK


---
### 1. Carregamento dos Dados

#### 1.1 Ranking Parcial (NB09)

In [2]:
ranking = pd.read_csv('data/output/ranking_parcial.csv')

ranking['cnpj_str'] = ranking['cnpj_forn_norm'].astype(str).str.zfill(14)

print(f'Shape: {ranking.shape}')
print(f'Colunas: {ranking.columns.tolist()}')
print()
print(ranking[['posicao_risco', 'cnpj_str', 'votos', 'score_combinado']].head(5))

Shape: (75471, 18)
Colunas: ['orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'cnpj_forn_norm', 'if_score', 'if_label', 'if_score_norm', 'lof_score', 'lof_label', 'lof_score_norm', 'ocsvm_score', 'ocsvm_label', 'ocsvm_score_norm', 'votos', 'score_combinado', 'posicao_risco', 'cnpj_str']

   posicao_risco        cnpj_str  votos  score_combinado
0              1  08961997000158      3           0.8585
1              2  52105018000151      2           0.5764
2              3  51026267000199      2           0.5465
3              4  51026267000199      2           0.5446
4              5  50290552000150      2           0.5306


#### 1.2 CEIS — Cadastro de Empresas Inidôneas e Suspensas

In [3]:
ceis = pd.read_csv('data/raw/dados_CEIS.csv', sep=';', encoding='latin1')

print(f'Shape: {ceis.shape}')
print(f'Tipo de pessoa: {ceis["TIPO DE PESSOA"].value_counts().to_dict()}')

Shape: (23706, 24)
Tipo de pessoa: {'J': 14533, 'F': 9161}


---
### 2. Limpeza dos Dados

#### 2.1 Filtro: Apenas Pessoas Jurídicas + Limpeza de CNPJs

In [4]:
ceis_pj = ceis[ceis['TIPO DE PESSOA'] == 'J'].copy()

ceis_pj['cnpj_clean'] = (
    ceis_pj['CPF OU CNPJ DO SANCIONADO']
    .astype(str)
    .str.replace(r'\D', '', regex=True)
    .str.zfill(14)
)

print(f'PJ total: {len(ceis_pj):,}  |  CNPJs únicos: {ceis_pj["cnpj_clean"].nunique():,}')

PJ total: 14,533  |  CNPJs únicos: 9,606


#### 2.2 Separação: Sanções Ativas vs. Histórico Completo

- `cnpjs_ceis_todos`: Todos os CNPJs sancionados historicamente (2003–2026) 
- `cnpjs_ceis_ativos`: CNPJs com sanção ativa em 2026-10-05 (data final ≥ hoje ou indefinida) 

In [5]:
ceis_pj['DATA INÍCIO SANÇÃO'] = pd.to_datetime(
    ceis_pj['DATA INÍCIO SANÇÃO'], format='%d/%m/%Y', errors='coerce'
)
ceis_pj['DATA FINAL SANÇÃO'] = pd.to_datetime(
    ceis_pj['DATA FINAL SANÇÃO'], format='%d/%m/%Y', errors='coerce'
)

DATA_REFERENCIA = pd.Timestamp('2026-10-05')

ceis_ativos = ceis_pj[
    ceis_pj['DATA FINAL SANÇÃO'].isna() |
    (ceis_pj['DATA FINAL SANÇÃO'] >= DATA_REFERENCIA)
]

cnpjs_ceis_todos  = set(ceis_pj['cnpj_clean'].unique())
cnpjs_ceis_ativos = set(ceis_ativos['cnpj_clean'].unique())

print(f'CNPJs CEIS — histórico completo:  {len(cnpjs_ceis_todos):,}')
print(f'CNPJs CEIS — sanção ativa:        {len(cnpjs_ceis_ativos):,}')

CNPJs CEIS — histórico completo:  9,606
CNPJs CEIS — sanção ativa:        9,108


---
### 3. Cruzamento Ranking × CEIS

In [6]:
ranking['no_ceis_total'] = ranking['cnpj_str'].isin(cnpjs_ceis_todos).astype(int)
ranking['no_ceis_ativo']  = ranking['cnpj_str'].isin(cnpjs_ceis_ativos).astype(int)

print('=== Resultado do Cruzamento ===')
print()
print(f'Total de propostas:                 {len(ranking):,}')
print(f'CNPJs únicos no dataset:            {ranking["cnpj_str"].nunique():,}')
print(f'CNPJs únicos no CEIS (histórico):   {len(cnpjs_ceis_todos):,}')
print()
n_tot = ranking['no_ceis_total'].sum()
n_ati = ranking['no_ceis_ativo'].sum()
print(f'Propostas c/ fornecedor no CEIS (histórico): {n_tot:,}  ({n_tot/len(ranking)*100:.2f}%)')
print(f'Propostas c/ fornecedor no CEIS (ativo):     {n_ati:,}  ({n_ati/len(ranking)*100:.2f}%)')
print()
print('--- Por número de votos ---')
print(ranking.groupby('votos')[['no_ceis_total', 'no_ceis_ativo']].agg(['sum', 'mean']).round(4))

=== Resultado do Cruzamento ===

Total de propostas:                 75,471
CNPJs únicos no dataset:            9,437
CNPJs únicos no CEIS (histórico):   9,606

Propostas c/ fornecedor no CEIS (histórico): 11,674  (15.47%)
Propostas c/ fornecedor no CEIS (ativo):     11,308  (14.98%)

--- Por número de votos ---
      no_ceis_total        no_ceis_ativo       
                sum   mean           sum   mean
votos                                          
0             11513 0.1566         11148 0.1516
1               155 0.0972           154 0.0966
2                 6 0.0168             6 0.0168
3                 0 0.0000             0 0.0000


---
### 4. Discussão

Resultado observado: apenas 5 das 424 propostas de alto risco (≥ 2 votos) têm fornecedor com sanção ativa no CEIS — taxa de 1,2%, inferior aos 15% da base completa.

Esse resultado não invalida os modelos — ao contrário, é consistente com a natureza da detecção de anomalias em contextos sem supervisão:

| Hipótese | Explicação |
|---|---|
| **Detecção prospectiva** | O CEIS registra empresas já *formalmente* sancionadas. Os modelos podem estar identificando empresas com padrões suspeitos que ainda não foram auditadas. |
| **Perfil diferente de anomalia** | As features analíticas (preço, competição, temporalidade) capturam padrões distintos dos que levam a sanções formais no CEIS (que frequentemente envolvem fraude documental ou conluio identificado em processo administrativo). |
| **Limitação do CEIS** | Apenas uma fração das irregularidades em licitações resulta em processos formais com sanção publicada. A maioria dos casos suspeitos nunca chega ao CEIS. |

In [7]:
alto_risco = ranking[ranking['votos'] >= 2].copy()

print(f'Propostas de alto risco (votos ≥ 2): {len(alto_risco):,}')
print(f'  → CEIS ativo: {alto_risco["no_ceis_ativo"].sum()} ({alto_risco["no_ceis_ativo"].mean()*100:.1f}%)')
print()

baseline_rate = ranking['no_ceis_ativo'].mean()
expected_random = baseline_rate * len(alto_risco)
print(f'Taxa de CEIS na base completa (baseline): {baseline_rate*100:.1f}%')
print(f'Hits esperados por seleção aleatória:     {expected_random:.0f}')
print(f'Hits observados:                          {alto_risco["no_ceis_ativo"].sum()}')
print()

for v in [3, 2]:
    sub = ranking[ranking['votos'] == v]
    print(f'votos={v} ({len(sub)} propostas): {sub["no_ceis_ativo"].sum()} no CEIS ativo ({sub["no_ceis_ativo"].mean()*100:.1f}%)')

Propostas de alto risco (votos ≥ 2): 359
  → CEIS ativo: 6 (1.7%)

Taxa de CEIS na base completa (baseline): 15.0%
Hits esperados por seleção aleatória:     54
Hits observados:                          6

votos=3 (1 propostas): 0 no CEIS ativo (0.0%)
votos=2 (358 propostas): 6 no CEIS ativo (1.7%)
